In [1]:
import pandas as pd

us = pd.read_csv('../data/processed/user_summary_clustered.csv')

In [2]:
cluster_order = us.groupby('Cluster')['AvgSteps'].mean().sort_values().index.tolist()
n = len(cluster_order)

if n == 2:
    names = ['Sedentary User', 'Active User']
elif n == 3:
    names = ['Sedentary User', 'Moderate Mover', 'High Intensity User']
elif n == 4:
    names = ['Sedentary User', 'Light Active', 'Steady Mover', 'High Intensity User']
else:
    names = [f'Level {i+1} User' for i in range(n)]

persona_map = dict(zip(cluster_order, names))
us['Persona'] = us['Cluster'].map(persona_map)

In [3]:
# 주말>평일 비중이 뚜렷한 클러스터는 Weekend Warrior로 별도 태깅

weekend_ratio = us.groupby('Cluster').apply(lambda g: g['AvgSteps_Weekend'].mean() / g['AvgSteps_Weekday'].mean())
weekend_heavy = weekend_ratio[weekend_ratio > 1.15].index.tolist()
us.loc[us['Cluster'].isin(weekend_heavy), 'Persona'] = us.loc[us['Cluster'].isin(weekend_heavy), 'Persona'] + ' (Weekend Warrior 경향)'

In [4]:
summary = us.groupby('Persona').agg(
    n=('Cluster','size'),
    AvgSteps=('AvgSteps','mean'),
    AvgActiveMinutes=('AvgActiveMinutes','mean'),
    AvgVeryActiveMinutes=('AvgVeryActiveMinutes','mean'),
    AvgSteps_Weekday=('AvgSteps_Weekday','mean'),
    AvgSteps_Weekend=('AvgSteps_Weekend','mean'),
).round(1)

us.to_csv('../data/processed/user_summary_clustered.csv', index=False)
summary

,n,AvgSteps,AvgActiveMinutes,AvgVeryActiveMinutes,AvgSteps_Weekday,AvgSteps_Weekend
Persona,,,,,,
Active User,22,9391.5,276.5,27.8,9402.5,9399.7
Sedentary User,11,4589.3,160.4,6.6,4623.5,4589.1
